# Day 4 & 5: PersonaPath App & Gradio UI
Inference with Wav2Vec 2.0, CNN, LSTM, and Claude API, featuring Gradio visualizations.

In [ ]:
!pip install transformers librosa pronouncing python-Levenshtein gradio anthropic opencv-python huggingface_hub matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import gradio as gr
import librosa
import librosa.display
import pronouncing
import Levenshtein
import torch
import cv2
import numpy as np
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import anthropic
from huggingface_hub import from_pretrained_keras

VISUALIZATION_DIR = '/content/drive/MyDrive/PersonaPath/visualizations'
os.makedirs(VISUALIZATION_DIR, exist_ok=True)
print(f'Ensured visualization directory exists at: {VISUALIZATION_DIR}')

ANTHROPIC_API_KEY = 'YOUR_API_KEY_HERE'
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
# Load Models
print('Loading Wav2Vec 2.0...')
processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
model_w2v = Wav2Vec2ForCTC.from_pretrained('facebook/wav2vec2-base-960h')

print('--- Load Phase 2 Models ---')
cnn_repo_id = input('Enter your CNN Emotion HF repo ID (or leave blank to skip): ')
cnn_model = from_pretrained_keras(cnn_repo_id) if cnn_repo_id else None

lstm_repo_id = input('Enter your LSTM Confidence HF repo ID (or leave blank to skip): ')
lstm_model = from_pretrained_keras(lstm_repo_id) if lstm_repo_id else None

print('Models loaded successfully!')

In [ ]:
def phase_1_analysis(audio_path, reference_text):
    if not audio_path or not reference_text:
        return "", "", "", None
    
    audio_input, sr = librosa.load(audio_path, sr=16000)
    
    # VISUALIZATION: Plot Waveform and save to Drive
    plt.figure(figsize=(10, 4))
    librosa.display.waveshow(audio_input, sr=sr, color='blue', alpha=0.5)
    plt.title('Recorded Speech Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    wave_path = os.path.join(VISUALIZATION_DIR, 'phase1_inference_waveform.png')
    plt.savefig(wave_path)
    plt.close()
    
    inputs = processor(audio_input, sampling_rate=sr, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = model_w2v(inputs.input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0].lower()
    
    ref_words = reference_text.lower().split()
    trans_words = transcription.split()
    
    highlighted_text = ''
    for ref in ref_words:
        distances = [Levenshtein.distance(ref, t) for t in trans_words]
        min_dist = min(distances) if distances else len(ref)
        if min_dist == 0:
            highlighted_text += f'<span style="color: green; font-weight: bold;">{ref}</span> '
        elif min_dist <= 2:
            highlighted_text += f'<span style="color: red; font-weight: bold;">{ref}</span> '
        else:
            highlighted_text += f'<span style="color: gray; text-decoration: line-through;">{ref}</span> '
            
    duration_mins = len(audio_input) / sr / 60.0
    wpm = len(trans_words) / duration_mins if duration_mins > 0 else 0
    
    prompt = f'A child read: "{reference_text}". WPM: {wpm:.1f}. Give a 1-sentence encouraging tip for a 7-year-old.'
    try:
        response = claude_client.messages.create(
            model="claude-3-haiku-20240307", max_tokens=50, messages=[{"role": "user", "content": prompt}]
        )
        tip = response.content[0].text
    except Exception as e:
        tip = "Keep practicing! You are doing great."

    return f'<h3>{highlighted_text}</h3>', f'{wpm:.1f}', tip, wave_path

In [ ]:
def extract_frames(video_path, fps=3):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_rate = int(cap.get(cv2.CAP_PROP_FPS))
    interval = max(int(frame_rate / fps), 1)
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if count % interval == 0:
            frame = cv2.resize(frame, (224, 224))
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        count += 1
    cap.release()
    return np.array(frames)

def phase_2_analysis(video_path):
    if not video_path:
        return "", "", "", None
    
    emotion = 'Confident'
    confidence_score = 75.0
    emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
    avg_preds = [0.1, 0.05, 0.05, 0.2, 0.05, 0.15, 0.4] # Default mock data
    
    if cnn_model:
        frames = extract_frames(video_path)
        if len(frames) > 0:
            frames_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(frames.astype(np.float32))
            preds = cnn_model.predict(frames_preprocessed)
            avg_preds = np.mean(preds, axis=0)
            emotion = emotions[np.argmax(avg_preds)]

    if lstm_model:
        audio_input, sr = librosa.load(video_path, sr=22050)
        mfcc = librosa.feature.mfcc(y=audio_input, sr=sr, n_mfcc=13).T
        if mfcc.shape[0] < 200:
            pad_width = 200 - mfcc.shape[0]
            mfcc = np.pad(mfcc, pad_width=((0, pad_width), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:200, :]
        conf_pred = lstm_model.predict(np.expand_dims(mfcc, axis=0))
        confidence_score = float(conf_pred[0][0])
    
    # VISUALIZATION: Plot Emotion Probabilities and save to Drive
    plt.figure(figsize=(8, 4))
    sns.barplot(x=emotions, y=avg_preds, palette='viridis')
    plt.title('Emotion Probabilities per Presentation')
    plt.ylabel('Probability')
    plt.tight_layout()
    chart_path = os.path.join(VISUALIZATION_DIR, 'phase2_emotion_chart.png')
    plt.savefig(chart_path)
    plt.close()

    prompt = f'A 12-year-old student presented. Confidence: {confidence_score:.1f}%, Emotion: {emotion}. Give actionable public speaking advice.'
    try:
        response = claude_client.messages.create(
            model="claude-3-haiku-20240307", max_tokens=50, messages=[{"role": "user", "content": prompt}]
        )
        tip = response.content[0].text
    except:
        tip = "Maintain eye contact and take deep breaths."
        
    return f'{confidence_score:.1f}%', emotion, tip, chart_path

In [ ]:
# Gradio Dashboard with Integrated Visualizations
with gr.Blocks(title="PersonaPath", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ PersonaPath: Multi-Modal AI for Speaking Assessment")
    
    with gr.Tab("Phase 1: Literacy Coach (Ages 5-10)"):
        with gr.Row():
            with gr.Column():
                ref_text = gr.Textbox(label="Reference Text", value="The elephant sat quietly by the river bank.")
                audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath")
                analyze_btn = gr.Button("Analyze Pronunciation", variant="primary")
            with gr.Column():
                highlight_out = gr.HTML(label="Highlighted Speech (Green=Correct, Red=Mispronounced, Gray=Skipped)")
                wpm_out = gr.Textbox(label="Words Per Minute (WPM)")
                claude_out = gr.Textbox(label="Claude Coaching Tip")
        wave_out = gr.Image(label="Speech Waveform Analysis (Saved to Drive)")
        analyze_btn.click(phase_1_analysis, inputs=[audio_in, ref_text], outputs=[highlight_out, wpm_out, claude_out, wave_out])
        
    with gr.Tab("Phase 2: Presentation Pro (Ages 10+)"):
        with gr.Row():
            with gr.Column():
                video_in = gr.Video(label="Upload 30s Presentation")
                analyze_vid_btn = gr.Button("Analyze Presentation", variant="primary")
            with gr.Column():
                conf_out = gr.Textbox(label="Overall Confidence Score (0-100)")
                emot_out = gr.Textbox(label="Primary Emotion")
                claude_vid_out = gr.Textbox(label="Claude Coaching Tip")
        emotion_plot_out = gr.Image(label="Emotion Probabilities Distribution (Saved to Drive)")
        analyze_vid_btn.click(phase_2_analysis, inputs=[video_in], outputs=[conf_out, emot_out, claude_vid_out, emotion_plot_out])

demo.launch(debug=True)